In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. قراءة البيانات
print("Loading Data...")
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

print("=== 1. Content-Based Filtering ===")
# استخدام التصنيفات لإيجاد الأفلام المتشابهة
movies['genres'] = movies['genres'].str.replace('|', ' ')
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres'])
content_similarity = cosine_similarity(tfidf_matrix, tfidf_matrix)

def get_content_recommendations(title, cosine_sim=content_similarity):
    idx = movies.index[movies['title'].str.contains(title, case=False, na=False)].tolist()[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:6]
    movie_indices = [i[0] for i in sim_scores]
    return movies['title'].iloc[movie_indices].tolist()

target_movie = "Toy Story"
print(f"Recommendations based on the genres of '{target_movie}':")
for i, rec in enumerate(get_content_recommendations(target_movie), 1):
    print(f"{i}. {rec}")


print("\n=== 2. Collaborative Filtering (Item-Based) ===")
# إنشاء جدول يجمع المستخدمين والأفلام
user_movie_matrix = ratings.merge(movies, on='movieId').pivot_table(index='userId', columns='title', values='rating').fillna(0)

# أخذ الأفلام المشهورة فقط لتسريع العملية
ratings_count = pd.DataFrame(ratings.groupby('movieId')['rating'].count())
popular_movies = ratings_count[ratings_count['rating'] >= 50].index
popular_movies_titles = movies[movies['movieId'].isin(popular_movies)]['title']
user_movie_matrix = user_movie_matrix[popular_movies_titles]

# حساب التشابه بناءً على تقييمات المستخدمين
item_similarity = cosine_similarity(user_movie_matrix.T)
item_similarity_df = pd.DataFrame(item_similarity, index=user_movie_matrix.columns, columns=user_movie_matrix.columns)

def get_collaborative_recommendations(title):
    similar_scores = item_similarity_df[title].sort_values(ascending=False)
    return similar_scores.iloc[1:6].index.tolist()

target_collab_movie = "Matrix, The (1999)"
print(f"Users who liked '{target_collab_movie}' also liked:")
for i, rec in enumerate(get_collaborative_recommendations(target_collab_movie), 1):
    print(f"{i}. {rec}")

Loading Data...
=== 1. Content-Based Filtering ===
Recommendations based on the genres of 'Toy Story':
1. Antz (1998)
2. Toy Story 2 (1999)
3. Adventures of Rocky and Bullwinkle, The (2000)
4. Emperor's New Groove, The (2000)
5. Monsters, Inc. (2001)

=== 2. Collaborative Filtering (Item-Based) ===
Users who liked 'Matrix, The (1999)' also liked:
1. Fight Club (1999)
2. Star Wars: Episode V - The Empire Strikes Back (1980)
3. Saving Private Ryan (1998)
4. Star Wars: Episode IV - A New Hope (1977)
5. Star Wars: Episode VI - Return of the Jedi (1983)
